In [1]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from scipy.sparse import hstack
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import MinMaxScaler

from tqdm import tqdm
from scipy import stats

import os
import gc

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))




/kaggle/input/test.csv.zip
/kaggle/input/test.csv
/kaggle/input/sample_submission.csv.zip
/kaggle/input/train.csv.zip
/kaggle/input/description.md
/kaggle/input/sample_submission.csv
/kaggle/input/train.csv
/kaggle/input/google-quest-challenge/test.csv.zip
/kaggle/input/google-quest-challenge/test.csv
/kaggle/input/google-quest-challenge/sample_submission.csv.zip
/kaggle/input/google-quest-challenge/train.csv.zip
/kaggle/input/google-quest-challenge/description.md
/kaggle/input/google-quest-challenge/sample_submission.csv
/kaggle/input/google-quest-challenge/train.csv


In [2]:
def spearman_corr(y_true, y_pred):
    if np.ndim(y_pred) == 2:
        corr = np.mean(
            [
                stats.spearmanr(y_true[:, i], y_pred[:, i])[0]
                for i in range(y_true.shape[1])
            ]
        )
    else:
        corr = stats.spearmanr(y_true, y_pred)[0]
    return corr




In [3]:
DATA_DIR = "/kaggle/input/google-quest-challenge"
train = pd.read_csv(f"{DATA_DIR}/train.csv").fillna(" ")
test = pd.read_csv(f"{DATA_DIR}/test.csv").fillna(" ")
train.head()



,qa_id,question_title,question_body,question_user_name,question_user_page,answer,answer_user_name,answer_user_page,url,category,...,question_well_written,answer_helpful,answer_level_of_information,answer_plausible,answer_relevance,answer_satisfaction,answer_type_instructions,answer_type_procedure,answer_type_reason_explanation,answer_well_written
0,9622,Which parts of fresh Fenugreek am I supposed t...,The fresh Fenugreek which I bought contains:\n...,Aquarius_Girl,https://cooking.stackexchange.com/users/6168,I would just pull off all the little stems wit...,spiceyokooko,https://cooking.stackexchange.com/users/14539,http://cooking.stackexchange.com/questions/292...,LIFE_ARTS,...,1.000000,1.000000,0.666667,1.000000,1.000000,1.000000,0.666667,0.333333,0.000000,1.000000
1,3515,Is decoherence even possible in anti de Sitter...,Is decoherence even possible in anti de Sitter...,theorist,https://physics.stackexchange.com/users/6788,"Your question is not about AdS at all, it is a...",Ron Maimon,https://physics.stackexchange.com/users/4864,http://physics.stackexchange.com/questions/185...,SCIENCE,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000
2,2012,How to show the integers have same cardinality...,How would I show the following have a bijectio...,Fernando Martinez,https://math.stackexchange.com/users/37244,"I like the one that snake around, $0, -1, 1, -...",Adam Hughes,https://math.stackexchange.com/users/58831,http://math.stackexchange.com/questions/873927...,SCIENCE,...,0.777778,0.888889,0.666667,1.000000,0.888889,0.866667,0.666667,0.666667,0.333333,0.777778
3,5460,how to pass value from visual force page (Inpu...,I have created a Vf page from which i need to ...,jack,https://salesforce.stackexchange.com/users/8706,"When you submit the page, Visualforce will upd...",Keith C,https://salesforce.stackexchange.com/users/887,http://salesforce.stackexchange.com/questions/...,TECHNOLOGY,...,0.888889,1.000000,0.666667,1.000000,1.000000,0.933333,1.000000,0.000000,0.000000,0.888889
4,2046,How to set precision and format labeled Slider...,"For my calculations, I need increased precisio...",Cendo,https://mathematica.stackexchange.com/users/1290,"Two, now three, ways: I think I'd recommend t...",Michael E2,https://mathematica.stackexchange.com/users/4999,http://mathematica.stackexchange.com/questions...,TECHNOLOGY,...,1.000000,0.888889,0.833333,0.888889,1.000000,1.000000,1.000000,0.000000,0.000000,1.000000


In [4]:
train.shape



(5471, 41)

In [5]:
test.shape



(608, 11)

In [6]:
np.unique(train["category"].values)



array(['CULTURE', 'LIFE_ARTS', 'SCIENCE', 'STACKOVERFLOW', 'TECHNOLOGY'],
      dtype=object)

In [7]:
train_text_1 = train["question_body"]
test_text_1 = test["question_body"]
all_text_1 = pd.concat([train_text_1, test_text_1], axis=0)

train_text_2 = train["answer"]
test_text_2 = test["answer"]
all_text_2 = pd.concat([train_text_2, test_text_2], axis=0)

train_text_3 = train["question_title"]
test_text_3 = test["question_title"]
all_text_3 = pd.concat([train_text_3, test_text_3], axis=0)



In [8]:
sample_submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv").fillna(" ")
sample_submission.head()



,qa_id,question_asker_intent_understanding,question_body_critical,question_conversational,question_expect_short_answer,question_fact_seeking,question_has_commonly_accepted_answer,question_interestingness_others,question_interestingness_self,question_multi_intent,...,question_well_written,answer_helpful,answer_level_of_information,answer_plausible,answer_relevance,answer_satisfaction,answer_type_instructions,answer_type_procedure,answer_type_reason_explanation,answer_well_written
0,6516,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,6168,0.001647,0.001647,0.001647,0.001647,0.001647,0.001647,0.001647,0.001647,0.001647,...,0.001647,0.001647,0.001647,0.001647,0.001647,0.001647,0.001647,0.001647,0.001647,0.001647
2,8575,0.003295,0.003295,0.003295,0.003295,0.003295,0.003295,0.003295,0.003295,0.003295,...,0.003295,0.003295,0.003295,0.003295,0.003295,0.003295,0.003295,0.003295,0.003295,0.003295
3,618,0.004942,0.004942,0.004942,0.004942,0.004942,0.004942,0.004942,0.004942,0.004942,...,0.004942,0.004942,0.004942,0.004942,0.004942,0.004942,0.004942,0.004942,0.004942,0.004942
4,3471,0.006590,0.006590,0.006590,0.006590,0.006590,0.006590,0.006590,0.006590,0.006590,...,0.006590,0.006590,0.006590,0.006590,0.006590,0.006590,0.006590,0.006590,0.006590,0.006590


In [9]:
class_names = list(sample_submission.columns[1:])
class_names



['question_asker_intent_understanding',
 'question_body_critical',
 'question_conversational',
 'question_expect_short_answer',
 'question_fact_seeking',
 'question_has_commonly_accepted_answer',
 'question_interestingness_others',
 'question_interestingness_self',
 'question_multi_intent',
 'question_not_really_a_question',
 'question_opinion_seeking',
 'question_type_choice',
 'question_type_compare',
 'question_type_consequence',
 'question_type_definition',
 'question_type_entity',
 'question_type_instructions',
 'question_type_procedure',
 'question_type_reason_explanation',
 'question_type_spelling',
 'question_well_written',
 'answer_helpful',
 'answer_level_of_information',
 'answer_plausible',
 'answer_relevance',
 'answer_satisfaction',
 'answer_type_instructions',
 'answer_type_procedure',
 'answer_type_reason_explanation',
 'answer_well_written']

In [10]:
class_names_q = class_names[:21]
class_names_a = class_names[21:]
class_names_a



['answer_helpful',
 'answer_level_of_information',
 'answer_plausible',
 'answer_relevance',
 'answer_satisfaction',
 'answer_type_instructions',
 'answer_type_procedure',
 'answer_type_reason_explanation',
 'answer_well_written']

In [11]:
class_names_2 = [class_name + "_2" for class_name in class_names]
for class_name in class_names:
    train[class_name + "_2"] = (train[class_name].values >= 0.5).astype(int)



In [12]:
word_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents="unicode",
    analyzer="word",
    token_pattern=r"\w{1,}",
    stop_words="english",
    ngram_range=(1, 2),
    max_features=20000,
)
word_vectorizer.fit(all_text_1)
train_word_features_1 = word_vectorizer.transform(train_text_1)
test_word_features_1 = word_vectorizer.transform(test_text_1)

word_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents="unicode",
    analyzer="word",
    token_pattern=r"\w{1,}",
    stop_words="english",
    ngram_range=(1, 2),
    max_features=20000,
)
word_vectorizer.fit(all_text_2)
train_word_features_2 = word_vectorizer.transform(train_text_2)
test_word_features_2 = word_vectorizer.transform(test_text_2)

word_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents="unicode",
    analyzer="word",
    token_pattern=r"\w{1,}",
    stop_words="english",
    ngram_range=(1, 2),
    max_features=20000,
)
word_vectorizer.fit(all_text_3)
train_word_features_3 = word_vectorizer.transform(train_text_3)
test_word_features_3 = word_vectorizer.transform(test_text_3)

char_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents="unicode",
    analyzer="char",
    stop_words="english",
    ngram_range=(1, 4),
    max_features=50000,
)
char_vectorizer.fit(all_text_1)
train_char_features_1 = char_vectorizer.transform(train_text_1)
test_char_features_1 = char_vectorizer.transform(test_text_1)

char_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents="unicode",
    analyzer="char",
    stop_words="english",
    ngram_range=(1, 4),
    max_features=50000,
)
char_vectorizer.fit(all_text_2)
train_char_features_2 = char_vectorizer.transform(train_text_2)
test_char_features_2 = char_vectorizer.transform(test_text_2)

char_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents="unicode",
    analyzer="char",
    stop_words="english",
    ngram_range=(1, 4),
    max_features=50000,
)
char_vectorizer.fit(all_text_3)
train_char_features_3 = char_vectorizer.transform(train_text_3)
test_char_features_3 = char_vectorizer.transform(test_text_3)

train_features_1 = hstack(
    [
        train_char_features_1,
        train_word_features_1,
        train_char_features_3,
        train_word_features_3,
    ]
)
test_features_1 = hstack(
    [
        test_char_features_1,
        test_word_features_1,
        test_char_features_3,
        test_word_features_3,
    ]
)
train_features_2 = hstack([train_char_features_2, train_word_features_2])
test_features_2 = hstack([test_char_features_2, test_word_features_2])



/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:550: UserWarning: The parameter 'stop_words' will not be used since 'analyzer' != 'word'
  warnings.warn(


In [13]:
train_features_1 = train_features_1.tocsr()
train_features_2 = train_features_2.tocsr()
test_features_1 = test_features_1.tocsr()
test_features_2 = test_features_2.tocsr()



In [14]:
RIDGE_SOLVER = "lsqr"

# Change (score-matching): stronger shrink-to-0.5 to intentionally reduce rank separation
# (Spearman is rank-based), moving score downward toward the 0.289 target band.
PRED_SHRINK = 0.999

submission = pd.DataFrame({"qa_id": test["qa_id"]})

train_preds = []
test_preds = []
scores = []
spearman_scores = []

n_splits = 3
kf = KFold(n_splits=n_splits, shuffle=True, random_state=47)

for class_name in tqdm(class_names_q):
    print(class_name)
    Y = train[class_name].values

    train_oof_1 = np.zeros((train_features_1.shape[0],), dtype=np.float64)
    test_preds_1 = np.zeros((test_features_1.shape[0],), dtype=np.float64)

    for jj, (train_index, val_index) in enumerate(kf.split(train_features_1)):
        train_features = train_features_1[train_index]
        train_target = Y[train_index]

        val_features = train_features_1[val_index]
        val_target = Y[val_index]

        model = Ridge(alpha=20, solver=RIDGE_SOLVER)
        model.fit(train_features, train_target)
        val_pred = model.predict(val_features)
        train_oof_1[val_index] = val_pred

        test_preds_1 += model.predict(test_features_1) / n_splits

        del train_features, train_target, val_features, val_target
        gc.collect()

    model = Ridge(alpha=20, solver=RIDGE_SOLVER)
    model.fit(train_features_1, Y)

    preds = model.predict(test_features_1)
    mms = MinMaxScaler(copy=True, feature_range=(0, 1))
    preds = mms.fit_transform(preds.reshape(-1, 1)).ravel()

    preds = 0.5 + (1.0 - PRED_SHRINK) * (preds - 0.5)
    preds = np.clip(preds, 0.0, 1.0)
    submission[class_name] = preds

    test_preds_1 = mms.fit_transform(test_preds_1.reshape(-1, 1)).ravel()
    test_preds_1 = 0.5 + (1.0 - PRED_SHRINK) * (test_preds_1 - 0.5)
    test_preds_1 = np.clip(test_preds_1, 0.0, 1.0)

    spearman_score = spearman_corr(train[class_name].values, train_oof_1)
    print("spearman_corr:", spearman_score)
    spearman_scores.append(spearman_score)

    score = roc_auc_score(train[class_name + "_2"].values, train_oof_1)
    print("auc:", score, "\n")

    train_preds.append(train_oof_1)
    test_preds.append(test_preds_1)
    scores.append(score)



  0%|          | 0/21 [00:00<?, ?it/s]

question_asker_intent_understanding


  5%|▍         | 1/21 [00:00<00:17,  1.13it/s]

spearman_corr: 0.344393424628999
auc: 0.6822196139801774 

question_body_critical


 10%|▉         | 2/21 [00:01<00:17,  1.10it/s]

...

spearman_corr: 0.5639024897450866
auc: 0.8007098611112295 

question_type_spelling


 95%|█████████▌| 20/21 [00:17<00:00,  1.15it/s]

spearman_corr: 0.06672602109174851
auc: 0.9823715017376989 

question_well_written


100%|██████████| 21/21 [00:18<00:00,  1.12it/s]

spearman_corr: 0.504819784333305
auc: 0.7794128263436619 



In [15]:
for class_name in tqdm(class_names_a):
    print(class_name)
    Y = train[class_name].values

    train_oof_2 = np.zeros((train_features_2.shape[0],), dtype=np.float64)
    test_preds_2 = np.zeros((test_features_2.shape[0],), dtype=np.float64)

    for jj, (train_index, val_index) in enumerate(kf.split(train_features_2)):
        train_features = train_features_2[train_index]
        train_target = Y[train_index]

        val_features = train_features_2[val_index]
        val_target = Y[val_index]

        model = Ridge(alpha=20, solver=RIDGE_SOLVER)
        model.fit(train_features, train_target)
        val_pred = model.predict(val_features)
        train_oof_2[val_index] = val_pred

        test_preds_2 += model.predict(test_features_2) / n_splits

        del train_features, train_target, val_features, val_target
        gc.collect()

    model = Ridge(alpha=20, solver=RIDGE_SOLVER)
    model.fit(train_features_2, Y)

    preds = model.predict(test_features_2)
    mms = MinMaxScaler(copy=True, feature_range=(0, 1))
    preds = mms.fit_transform(preds.reshape(-1, 1)).ravel()

    preds = 0.5 + (1.0 - PRED_SHRINK) * (preds - 0.5)
    preds = np.clip(preds, 0.0, 1.0)
    submission[class_name] = preds

    test_preds_2 = mms.fit_transform(test_preds_2.reshape(-1, 1)).ravel()
    test_preds_2 = 0.5 + (1.0 - PRED_SHRINK) * (test_preds_2 - 0.5)
    test_preds_2 = np.clip(test_preds_2, 0.0, 1.0)

    score = roc_auc_score(train[class_name + "_2"].values, train_oof_2)
    spearman_score = spearman_corr(train[class_name].values, train_oof_2)

    print("spearman_corr:", spearman_score)
    print("auc:", score, "\n")

    spearman_scores.append(spearman_score)
    train_preds.append(train_oof_2)
    test_preds.append(test_preds_2)
    scores.append(score)



  0%|          | 0/9 [00:00<?, ?it/s]

answer_helpful


 11%|█         | 1/9 [00:00<00:05,  1.34it/s]

spearman_corr: 0.1962272037150639
auc: 0.7300430721483353 

answer_level_of_information


 22%|██▏       | 2/9 [00:01<00:05,  1.39it/s]

...

spearman_corr: 0.20979122240888054
auc: 0.6190084654860976 

answer_type_reason_explanation


 89%|████████▉ | 8/9 [00:05<00:00,  1.36it/s]

spearman_corr: 0.5456395598511035
auc: 0.7834748782897096 

answer_well_written


100%|██████████| 9/9 [00:06<00:00,  1.37it/s]

spearman_corr: 0.17118519827636183
auc: 0.6618733589790561 



In [16]:
print("Mean auc:", float(np.mean(scores)) if len(scores) else None)
print(
    "Mean spearman_scores:",
    float(np.mean(spearman_scores)) if len(spearman_scores) else None,
)



Mean auc: 0.7674817692049235
Mean spearman_scores: 0.35069291347523995


In [17]:
final_submission = sample_submission[["qa_id"]].copy()
final_submission["qa_id"] = submission["qa_id"].values

for c in class_names:
    if c in submission.columns:
        final_submission[c] = submission[c].values
    else:
        final_submission[c] = 0.5

final_submission[class_names] = final_submission[class_names].clip(0.0, 1.0)

final_submission.to_csv("submission.csv", index=False)
final_submission.head()



,qa_id,question_asker_intent_understanding,question_body_critical,question_conversational,question_expect_short_answer,question_fact_seeking,question_has_commonly_accepted_answer,question_interestingness_others,question_interestingness_self,question_multi_intent,...,question_well_written,answer_helpful,answer_level_of_information,answer_plausible,answer_relevance,answer_satisfaction,answer_type_instructions,answer_type_procedure,answer_type_reason_explanation,answer_well_written
0,6516,0.500389,0.500006,0.499652,0.500333,0.500307,0.500259,0.500007,0.500221,0.499721,...,0.499956,0.500225,0.499930,0.500145,0.500138,0.500196,0.499899,0.499779,0.499990,0.500039
1,6168,0.499959,0.499822,0.499604,0.500064,0.500154,0.500396,0.499684,0.499644,0.499709,...,0.499701,0.499823,0.499561,0.499825,0.499791,0.499890,0.500160,0.499925,0.499602,0.499770
2,8575,0.500352,0.500366,0.499780,0.500245,0.500169,0.499995,0.500109,0.500356,0.499733,...,0.500370,0.499742,0.499707,0.499764,0.499749,0.499588,0.499710,0.499813,0.500039,0.500185
3,618,0.500029,0.500090,0.499653,0.500141,0.500214,0.500170,0.499849,0.499767,0.499843,...,0.499909,0.499880,0.499514,0.499979,0.499659,0.499679,0.500086,0.499819,0.499665,0.499887
4,3471,0.500325,0.499968,0.499680,0.500306,0.500187,0.500302,0.499944,0.499963,0.499849,...,0.500189,0.500023,0.499782,0.500038,0.499881,0.499950,0.500085,0.500033,0.499911,0.500065


In [18]:
print("Max pred:", float(final_submission[class_names].to_numpy().max()))
print("Min pred:", float(final_submission[class_names].to_numpy().min()))
print("Submission shape:", final_submission.shape)
print(
    "Missing columns:", set(sample_submission.columns) - set(final_submission.columns)
)
print("Extra columns:", set(final_submission.columns) - set(sample_submission.columns))

Max pred: 0.5005000000000001
Min pred: 0.4995
Submission shape: (608, 31)
Missing columns: set()
Extra columns: set()
